In [1]:
import pandas as pd

df = pd.read_csv("../data/ChargingRecords.csv")

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())
print(df.describe())


(72856, 13)
   UserID  ChargerID  ChargerCompany Location  ChargerType    StartDay  \
0       0          1               1    hotel            0  2022-09-15   
1       0          1               1    hotel            0  2022-09-14   
2       0          1               1    hotel            0  2022-09-14   
3       0          1               1    hotel            0  2022-09-29   
4       0          1               1    hotel            0  2022-09-25   

  StartTime      EndDay   EndTime     StartDatetime       EndDatetime  \
0  20:54:02  2022-09-15  23:59:13  2022-09-15 20:54  2022-09-15 23:59   
1  20:01:05  2022-09-14  21:31:04  2022-09-14 20:01  2022-09-14 21:31   
2  18:54:30  2022-09-14  19:54:29  2022-09-14 18:54  2022-09-14 19:54   
3  18:32:51  2022-09-30   0:16:42  2022-09-29 18:32   2022-09-30 0:16   
4  19:30:15  2022-09-26   0:30:14  2022-09-25 19:30   2022-09-26 0:30   

   Duration  Demand  
0       185   20.36  
1        90   10.19  
2        60    6.78  
3       344   37

In [2]:
import numpy, sklearn, xgboost, lightgbm, shap, lime, imblearn, streamlit
print("All libraries imported successfully")

c:\Users\91903\Downloads\EV Charging Demand Prediction and Usage Pattern Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully


In [3]:
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

UserID              int64
ChargerID           int64
ChargerCompany      int64
Location              str
ChargerType         int64
StartDay              str
StartTime             str
EndDay                str
EndTime               str
StartDatetime         str
EndDatetime           str
Duration            int64
Demand            float64
dtype: object

Missing values:
UserID            0
ChargerID         0
ChargerCompany    0
Location          0
ChargerType       0
StartDay          0
StartTime         0
EndDay            0
EndTime           0
StartDatetime     0
EndDatetime       0
Duration          0
Demand            0
dtype: int64

Duplicate rows: 13


In [4]:
print("Unique users:", df["UserID"].nunique())
print("Unique chargers:", df["ChargerID"].nunique())
print("\nLocations:\n", df["Location"].value_counts())
print("\nCharger companies:\n", df["ChargerCompany"].value_counts())
print("\nCharger types:\n", df["ChargerType"].value_counts())

Unique users: 2337
Unique chargers: 2119

Locations:
 Location
public area           14082
apartment             14038
resort                 8854
hotel                  8600
company                7558
public institution     7023
public parking lot     3396
market                 3005
restaurant             2222
camping                1630
sightseeing            1203
golf                   1029
accommodation           187
bus garage               29
Name: count, dtype: int64

Charger companies:
 ChargerCompany
1    45713
0    27143
Name: count, dtype: int64

Charger types:
 ChargerType
0    57903
1    14953
Name: count, dtype: int64


In [5]:
print(df["StartDay"].min(), "to", df["StartDay"].max())
print("\nSessions with Demand <= 0:", (df["Demand"] <= 0).sum())
print("Sessions with Duration <= 0:", (df["Duration"] <= 0).sum())
print("\nDemand percentiles:")
print(df["Demand"].describe(percentiles=[0.01, 0.5, 0.99]))

2021-09-30 to 2022-09-30

Sessions with Demand <= 0: 0
Sessions with Duration <= 0: 735

Demand percentiles:
count    72856.000000
mean        17.444445
std         13.492728
min          0.010000
1%           0.080000
50%         14.100000
99%         58.910000
max         97.000000
Name: Demand, dtype: float64


In [6]:
# Work on a copy so the original df stays untouched
df_clean = df.copy()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()
print("Rows after removing duplicates:", df_clean.shape[0])

# Build proper datetimes from the Day and Time columns.
# These include seconds, so they are more accurate than StartDatetime/EndDatetime.
df_clean["StartDatetime"] = pd.to_datetime(df_clean["StartDay"] + " " + df_clean["StartTime"])
df_clean["EndDatetime"] = pd.to_datetime(df_clean["EndDay"] + " " + df_clean["EndTime"])

# Recalculate duration in minutes to compare with the Duration column
df_clean["CalcDuration"] = (
    df_clean["EndDatetime"] - df_clean["StartDatetime"]
).dt.total_seconds() / 60

print(df_clean[["StartDatetime", "EndDatetime"]].dtypes)

Rows after removing duplicates: 72843
StartDatetime    datetime64[us]
EndDatetime      datetime64[us]
dtype: object


In [7]:
bad = df_clean[df_clean["Duration"] <= 0]

print("Rows with Duration <= 0:", len(bad))
print(bad[["StartDatetime", "EndDatetime", "Duration", "CalcDuration", "Demand"]].head(10))

print("\nDemand of these rows:")
print(bad["Demand"].describe())

print("\nDuration vs CalcDuration (all rows):")
print(df_clean[["Duration", "CalcDuration"]].describe())

Rows with Duration <= 0: 735
          StartDatetime         EndDatetime  Duration  CalcDuration  Demand
32  2022-06-10 18:52:53 2022-06-10 18:52:53         0      0.000000    0.54
33  2022-06-10 19:14:26 2022-06-10 19:14:26         0      0.000000    0.55
50  2022-03-01 12:45:35 2022-03-01 13:34:38         0     49.050000    5.58
57  2022-03-13 09:49:06 2022-03-13 10:41:42         0     52.600000    4.66
66  2022-02-11 08:45:14 2022-02-11 10:00:14         0     75.000000    8.34
106 2022-06-09 14:40:16 2022-06-09 14:40:39         0      0.383333    0.02
141 2022-08-01 11:14:23 2022-08-01 11:14:42         0      0.316667    0.02
244 2021-12-26 16:34:37 2021-12-26 17:01:55         0     27.300000    3.03
246 2021-12-27 17:29:49 2021-12-27 20:24:22         0    174.550000   17.32
254 2022-06-26 14:58:45 2022-06-26 14:59:14         0      0.483333    0.05

Demand of these rows:
count    735.000000
mean       4.775687
std       10.206069
min        0.010000
25%        0.020000
50%        0

In [8]:
# Type A: Duration says 0, but the real time gap is 1 minute or more
type_a = df_clean[(df_clean["Duration"] <= 0) & (df_clean["CalcDuration"] >= 1)]

# Type B: Duration says 0 and the real gap is also under 1 minute
type_b = df_clean[(df_clean["Duration"] <= 0) & (df_clean["CalcDuration"] < 1)]

print("Type A (fix Duration):", len(type_a))
print("Type B (tiny sessions):", len(type_b))
print("\nType B Demand:")
print(type_b["Demand"].describe())

Type A (fix Duration): 167
Type B (tiny sessions): 568

Type B Demand:
count    568.000000
mean       1.657412
std        6.305383
min        0.010000
25%        0.010000
50%        0.030000
75%        0.080000
max       53.240000
Name: Demand, dtype: float64


In [9]:
# Rows where the two duration columns disagree by more than 5 minutes
mismatch = df_clean[(df_clean["Duration"] - df_clean["CalcDuration"]).abs() > 5]
print("Rows where Duration and CalcDuration differ by > 5 min:", len(mismatch))

# Sessions longer than 24 hours
long_sessions = df_clean[df_clean["CalcDuration"] > 1440]
print("Sessions longer than 24 hours:", len(long_sessions))
print(long_sessions[["Duration", "CalcDuration", "Demand"]].head(10))

Rows where Duration and CalcDuration differ by > 5 min: 2176
Sessions longer than 24 hours: 21
       Duration  CalcDuration  Demand
2874        472   1444.850000    53.5
4862        229   2545.300000     8.0
4908        175   2363.433333     5.4
4911        167   2876.000000     5.1
6712        147   1731.716667    15.0
8022        132   3060.033333    14.4
8027        181   2468.133333    17.2
13420       157   3895.683333    16.7
18795       101   4006.283333     2.3
21530       214   2834.616667     7.2


In [10]:
# Keep only rows where Duration is positive (removes Type A and Type B)
df_clean = df_clean[df_clean["Duration"] > 0].copy()

# Sort by start time (useful later for time-based analysis)
df_clean = df_clean.sort_values("StartDatetime").reset_index(drop=True)

# Sanity checks
print("Final shape:", df_clean.shape)
print("Min Duration:", df_clean["Duration"].min())
print("Min Demand:", df_clean["Demand"].min())
print("Date range:", df_clean["StartDatetime"].min(), "to", df_clean["StartDatetime"].max())

# Save the cleaned dataset
df_clean.to_csv("../data/charging_clean.csv", index=False)
print("Saved to ../data/charging_clean.csv")

Final shape: (72108, 14)
Min Duration: 1
Min Demand: 0.01
Date range: 2021-09-30 17:14:26 to 2022-09-30 23:11:59
Saved to ../data/charging_clean.csv
